# Issue #251 — issue-slot and entity identity coverage

This notebook profiles the v2 issue → slot → play → 3/1/0 identity chain and keeps source-local, unresolved and unidentified entities separate. It does not open outcome, payout, or settlement-value columns.

In [1]:
from pathlib import Path
import json
import pandas as pd

root = Path('results/bj251_canonical_identity_evidence_v2')
manifest = json.loads((root / 'MANIFEST.json').read_text(encoding='utf-8'))
coverage = pd.read_csv(root / 'tables/coverage.csv')
ledger = pd.read_csv(root / 'tables/ambiguity_summary.csv')
aliases = pd.read_csv(root / 'tables/temporal_alias_summary.csv')
manifest

{'contract_version': 'BJ-UNIFIED-CANONICAL-IDENTITY-001-v2',
 'disposition': 'CANDIDATE_PENDING_PI_ACCEPTANCE',
 'exception_count': 36113,
 'external_data_root_env': 'BEIJING_LOT_DATA_ROOT',
 'external_package_relative': 'derived/bj251_canonical_identity_v2',
 'external_sha256sums_sha256': '44daf5904efc8e13b1099e4efc4a8e5aa826e47a3ca2906030d4fc9184ac66b0',
 'issue_count': 808,
 'issue_slot_count': 215193,
 'outcome_cell_count': 645579,
 'play_count': 215193,
 'trace_count': 236666}

In [2]:
coverage.groupby(['source_id', 'causal_grade'])[['rows', 'exact_slot_rows', 'resolved_entity_fields', 'source_local_entity_fields', 'conflicting_entity_fields', 'unidentified_entity_fields', 'unmatched_slot_rows']].sum()

,,rows,exact_slot_rows,resolved_entity_fields,source_local_entity_fields,conflicting_entity_fields,unidentified_entity_fields,unmatched_slot_rows
source_id,causal_grade,,,,,,,
beidan_history_db,SETTLEMENT_ONLY,38463,38463,31974,52843,30572,0,0
beidan_live_db,STRICT_CAUSAL,5494,5494,923,14940,223,396,0
beidan_xlsm_workbook,RETROSPECTIVE_PROXY,192709,192709,31051,514142,30349,2585,2560


In [3]:
ledger.groupby(['exception_class', 'disposition'])['rows'].sum().to_frame()

,,rows
exception_class,disposition,
CROSS_SOURCE_ENTITY_ALIAS_CONFLICT,UNRESOLVED,30572
MISSING_ENTITY_LABEL,UNIDENTIFIED,2981
MISSING_ISSUE_SLOT,QUARANTINED,2560


In [4]:
read_boundary = json.loads((root / 'READ_BOUNDARY.json').read_text(encoding='utf-8'))
case_policy = pd.read_csv(root / 'tables/exceptional_case_policy.csv')
print('Outcome-exposed inputs:', read_boundary['outcome_exposed_inputs'])
print('Fuzzy auto-acceptance:', read_boundary['fuzzy_auto_acceptance'])
print('Observed temporal aliases:', int(aliases['observed_alias_rows'].sum()))
case_policy

Outcome-exposed inputs: False
Fuzzy auto-acceptance: False
Observed temporal aliases: 6785


,case_class,disposition,rule
0,RESCHEDULED,MANUAL_REVIEW,retain issue slot but require provenance-bound...
1,POSTPONED,MANUAL_REVIEW,do not auto-map until a source declares a repl...
2,ABANDONED,EXCLUDE_FROM_INFERENCE,retain source trace; never create a settlement...
3,DUPLICATE,QUARANTINED,multiple source rows for one source and exact ...
4,NEUTRAL_VENUE,CANONICAL_NO_TEAM_REWRITE,venue does not alter home/away team identity; ...
5,RENAMED_ENTITY,MANUAL_REVIEW,only a provenance-bound temporal alias may con...
